In [29]:
import sys
sys.path.insert(0, '..')   # so Python can find src/

import pandas as pd
import numpy as np
import logging
import warnings
warnings.filterwarnings('ignore')

logging.basicConfig(
    filename='../pipeline.log',
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s'
)
logger = logging.getLogger('lab9')

In [30]:
# load data from CSV exported in Lab 8
# If the CSV does not exist yet, run the Lab 8 pipeline first
df_raw = pd.read_csv('../data/processed/analytics/articles.csv', low_memory=False)
print(f'Loaded {len(df_raw)} rows and {df_raw.shape[1]} columns')
df_raw.head(3)

Loaded 973 rows and 22 columns


,source_name,title,author,description,publishedAt,content,text,ID,Title,Source,...,Word Count,Category,Nutrition,Pharmacology,Mental Health,Total,raw_text,processed_text,published_date,image_url
0,Science Daily,Scientists discover diet that tricks the body ...,NaN,Researchers found that cutting two amino acids...,2026-02-27T18:05:43Z,"Shivering in the cold is uncomfortable, but it...",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Naturalnews.com,Nature’s powerhouses: The top 8 healthiest ber...,Belle Carter,"Berries are rich in vitamin C, fiber and polyp...",2026-03-12T06:00:00Z,"<ul><li>Berries are rich in vitamin C, fiber a...",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Eatingwell.com,6 Things You Should Do After 5 P.M. to Support...,Cheyenne Buckingham,Aging is a privilege—support your health in la...,2026-03-13T21:30:00Z,"Reviewed by Dietitian Sarah Pflugradt, Ph.D., ...",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [31]:
# Cell 3 - missing value report
from src.cleaning.missing_handler import report_missing

miss_report = report_missing(df_raw)
print(miss_report.to_string())

                missing_count  missing_pct    dtype
processed_text            972        99.90   object
raw_text                  972        99.90   object
Pharmacology              971        99.79  float64
Total                     971        99.79  float64
Mental Health             971        99.79  float64
Nutrition                 971        99.79  float64
ID                        958        98.46  float64
Title                     958        98.46   object
Source                    958        98.46   object
Author                    958        98.46   object
Category                  958        98.46   object
Published Date            955        98.15   object
Word Count                955        98.15   object
text                      952        97.84   object
published_date            925        95.07   object
image_url                 925        95.07   object
author                    177        18.19   object
content                    90         9.25   object
publishedAt 

In [32]:
# Save the report
miss_report.to_csv('../data/processed/cleaned/missing_report.csv')
print('Saved missing_report.csv')

Saved missing_report.csv


In [33]:
from src.cleaning.string_cleaner import (
    clean_title, clean_language_code, clean_description_text,
    extract_year_from_published_at
)

In [34]:
df = df_raw.copy()   # always work on a copy, keep the original safe
df = clean_title(df)
df = clean_language_code(df)
df = clean_description_text(df)
df = extract_year_from_published_at(df)

print('Columns after string cleaning:')
print(df[['title', 'description', 'publishedAt']].head(5).to_string())

Columns after string cleaning:
                                                                                   title                                                                                                                                                                                                                                                           description           publishedAt
0        Scientists discover diet that tricks the body into burning fat without exercise  Researchers found that cutting two amino acids common in animal protein—methionine and cysteine—made mice burn significantly more energy. The boost in heat production was nearly as powerful as constant exposure to cold temperatures. The mice didn’t eat less o…  2026-02-27T18:05:43Z
1       Nature’s powerhouses: The top 8 healthiest berries and their remarkable benefits  Berries are rich in vitamin C, fiber and polyphenols, which combat oxidative stress, inflammation and chronic diseases like heart dis

In [36]:
from src.analytics.regex_ops import (
    find_invalid_published_at,
    flag_short_descriptions
)

In [13]:
# Find date format problems
bad_dates = find_invalid_published_at(df)
print(f'Rows with bad date format: {bad_dates.sum()}')
print(df.loc[bad_dates, ['title', 'publishedAt']].head(5))

Rows with bad date format: 0
Empty DataFrame
Columns: [title, publishedAt]
Index: []


In [37]:
# Find short descriptions
short_ov = flag_short_descriptions(df, min_words=5)
print(f'\nArticles with very short descriptions: {short_ov.sum()}')
print(df.loc[short_ov, ['title', 'description']].head(5))


Articles with very short descriptions: 64
                                                 title  \
92    Tandem Diabetes Care Q4 Earnings Call Highlights   
95   Medtronic’s diabetes business MiniMed looks fo...   
124  This Critical Amino Acid May Increase Life Exp...   
156        Why Tandem Diabetes Care Stock Popped Today   
196  Medtronic plc (MDT) Gains Momentum Through Dia...   

                 description  
92                       NaN  
95                       NaN  
124  Don't sleep on taurine.  
156                      NaN  
196                      NaN  


In [38]:
from src.cleaning.deduplicator import (
    drop_exact_duplicates, drop_duplicate_ids, count_duplicates
)

In [39]:
print(f'Before deduplication: {len(df)} rows')

Before deduplication: 973 rows


In [40]:
# Step 1: drop completely identical rows
df = drop_exact_duplicates(df)
print(f'After exact dedup: {len(df)} rows')

After exact dedup: 367 rows


In [41]:
# Step 2: drop rows with the same url
dup_count_before = count_duplicates(df, col='url')
print(f'Duplicate urls found: {dup_count_before}')
df = drop_duplicate_ids(df, id_col='url')
print(f'After url dedup: {len(df)} rows')

Duplicate urls found: 0
After url dedup: 367 rows


In [42]:
# Cell 7 - type conversion
from src.cleaning.type_converter import (
    convert_dates, convert_numeric_columns,
    convert_category_columns, memory_report
)

In [43]:
df_before_types = df.copy()  # save snapshot for memory comparison

df = convert_dates(df)
df = convert_numeric_columns(df)
df = convert_category_columns(df)

print('\nData types after conversion:')
print(df.dtypes)


Data types after conversion:
source_name                  category
title                          object
author                         object
description                    object
publishedAt       datetime64[ns, UTC]
content                        object
text                           object
ID                            float64
Title                          object
Source                         object
Author                         object
Published Date                 object
Word Count                     object
Category                       object
Nutrition                     float64
Pharmacology                  float64
Mental Health                 float64
Total                         float64
raw_text                       object
processed_text                 object
published_date                 object
image_url                      object
published_year                  Int64
dtype: object


In [44]:
print('\nMemory comparison:')
memory_report(df_before_types, df)


Memory comparison:


In [45]:
from src.cleaning.missing_handler import drop_rows_missing_title
from src.cleaning.validator import run_all_validations

In [46]:
df = drop_rows_missing_title(df)

In [47]:
print('Running validation on cleaned DataFrame...')
run_all_validations(df)

Running validation on cleaned DataFrame...


In [48]:
print('\nSummary statistics of numeric columns:')
cols_to_check = ['published_year']
existing_cols = [col for col in cols_to_check if col in df.columns]
print(df[existing_cols].describe())


Summary statistics of numeric columns:
       published_year
count           297.0
mean           2026.0
std               0.0
min            2026.0
25%            2026.0
50%            2026.0
75%            2026.0
max            2026.0


In [49]:
print(df['title'].isna().sum())
print((df['title'].astype(str).str.strip() == '').sum())

0
0
